## Imports and Setup

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import base64
import mimetypes

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"OpenAI API key exists and begins {openai_api_key[:8]}")
else:
    print("OPENAI_API_KEY not found")
    
client = OpenAI()

## Model Choice

In [ ]:
MODEL_OPTIONS = [
    "gpt-4.1-mini",
    "gpt-4.1",
    "gpt-4o-mini",
]

DEFAULT_MODEL = "gpt-4.1-mini"

system_message = """
You are Study Buddy AI, a patient technical learning assistant.

Your job is to help the user understand programming, AI engineering,
tools, APIs, debugging, and course exercises.

Explain concepts simply, step by step.
Do not dump too much code at once.
When code is needed, explain the idea first, then provide the code.
If the user seems confused, slow down and use examples.
"""

## Helper Function

In [1]:
def image_file_to_data_url(file_path):
    mime_type, _ = mimetypes.guess_type(file_path)
    
    if mime_type is None:
        mime_type = "image/png"
    
    with open(file_path, "rb") as image_file:
        base64_image = base64.b64encode(image_file.read()).decode("utf-8")
    
    return f"data:{mime_type};base64,{base64_image}"

In [ ]:
def convert_gradio_part_to_openai_part(part):
    if part.get("type") == "text":
        return {
            "type": "text",
            "text": part.get("text", "")
        }

    if part.get("type") == "file":
        file_info = part.get("file", {})
        file_path = file_info.get("path")

        if file_path:
            mime_type, _ = mimetypes.guess_type(file_path)

            if mime_type and mime_type.startswith("image/"):
                image_data_url = image_file_to_data_url(file_path)

                return {
                    "type": "image_url",
                    "image_url": {
                        "url": image_data_url
                    }
                }

    return None

In [ ]:
def convert_gradio_history_to_openai_history(history):
    openai_history = []

    for message in history:
        role = message.get("role")
        content = message.get("content")

        if role == "assistant":
            openai_history.append({
                "role": "assistant",
                "content": content
            })

        elif role == "user":
            openai_content = []

            if isinstance(content, str):
                openai_content.append({
                    "type": "text",
                    "text": content
                })

            elif isinstance(content, list):
                for part in content:
                    converted_part = convert_gradio_part_to_openai_part(part)

                    if converted_part is not None:
                        openai_content.append(converted_part)

            openai_history.append({
                "role": "user",
                "content": openai_content
            })

    return openai_history

## Chat Function

In [ ]:
def chat(message, history, model):
    messages = [{"role": "system", "content": system_message}]

    clean_history = convert_gradio_history_to_openai_history(history)

    for item in clean_history:
        messages.append(item)

    user_content = []

    user_text = message.get("text", "")
    uploaded_files = message.get("files", [])

    if user_text:
        user_content.append({
            "type": "text",
            "text": user_text
        })

    for file_path in uploaded_files:
        mime_type, _ = mimetypes.guess_type(file_path)

        if mime_type and mime_type.startswith("image/"):
            image_data_url = image_file_to_data_url(file_path)

            user_content.append({
                "type": "image_url",
                "image_url": {
                    "url": image_data_url
                }
            })
        else:
            user_content.append({
                "type": "text",
                "text": f"[Uploaded file ignored because it is not an image: {file_path}]"
            })

    messages.append({
        "role": "user",
        "content": user_content
    })
    
    # ------- Check working ------------
    print("MESSAGES SENT TO OPENAI:")
    print(json.dumps(messages, indent=2)[:3000])
    # ------- Check working ------------

    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True
    )

    response = ""

    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        response += delta
        yield response